In [14]:
import kotlin.reflect.KMutableProperty
import kotlin.reflect.KMutableProperty0
import kotlin.reflect.KProperty

abstract class UniqueEntity {
    companion object {
        private var idCounter = 0
    }

    val id: Int by lazy { idCounter++ }
    val uuid: String by lazy { id.toString(36) }
}

typealias UpdateDomain<Type, Context> = Context.() -> Type?

operator fun <Type, Context> Context.invoke(tag: String, function: UpdateDomain<Type, Context>): Type? {
    TODO()
}

abstract class AbstractPropertyContext<T>(
    val tag: String?,
    val attributePool: MutableList<Pair<String, T?>>
) : UniqueEntity() {
    private var function: UpdateDomain<T, T?>? = null
    private fun getTag(property: KProperty<*>) = tag ?: property.name

    private fun get(tag: String): T? =
        attributePool.firstOrNull { it.first == tag }?.second

    private fun set(tag: String, value: T?) =
        if (!(attributePool.any { it.first == tag }))
            attributePool.add(tag to value)
        else
            attributePool.forEachIndexed { index, (key, _) ->
                if (key == tag) attributePool[index] = key to value
            }

    operator fun getValue(thisRef: Any?, property: KProperty<*>): UpdateDomain<T, T?>? = function

    operator fun setValue(thisRef: Any?, property: KProperty<*>, value: UpdateDomain<T, T?>?) {
        value?.let { value ->
            getTag(property).also { tag ->
                function = value
                set(tag, get(tag)(tag, value))
            }
        }
    }
}

/**
 * 属性上下文，提供属性委托功能
 * @param tag 属性标签，如果为null，则使用属性名作为标签
 * @param attributePool 属性池，存储属性标签和属性值的键值对
 */
class AttributeContext(tag: String?, attributePool: MutableList<Pair<String, String?>>) :
    AbstractPropertyContext<String?>(tag, attributePool)

val attributePool: MutableList<Pair<String, String?>> = mutableListOf()

fun attribute(tag: String): AttributeContext = AttributeContext(tag, attributePool)

In [15]:
var `data-text` by attribute("data-text")